# Setup
For dev, you must have the backend api running on your computer. For prod, please change USER_API_URL to reflect the production url.

In [393]:
import requests
import json
import os
import re
import pprint as pp
from dotenv import load_dotenv
from bson.objectid import ObjectId
from datetime import datetime
from functools import reduce
from pymongo import MongoClient, ReturnDocument, UpdateOne
from pymongo.errors import BulkWriteError

load_dotenv()
custom_request_header = os.getenv("CUSTOM_REQUEST_HEADER")
DATABASE_URL = os.getenv("DATABASE_URL")

# Connect to database and check current list of DBs

In [395]:
# Connect to MongoDB
client = MongoClient(DATABASE_URL)
print(client.list_database_names())

['backup_db', 'testdb', 'vrms-populate-projects-test', 'vrms-slack-dev', 'vrms-slack-main', 'vrms-slack-staging', 'vrms-test', 'vrms-test-2', 'vrms-test-3', 'vrms-test-4', 'vrms-test-5', 'vrms-test-6', 'vrms-test-clone-project-sync', 'vrms-test-copy', 'vrms-test-sync', 'vrms-user-migration-test', 'admin', 'local']


# Create a new test database

Define a source and copy for databases


In [396]:
db_source = client['vrms-test']
db_copy = client['vrms-populate-projects-test']

# Drop all collections in test database (ONLY IF NECESSARY!)


In [405]:
# for collection_name in db_copy.list_collection_names():
#     db_copy.drop_collection(collection_name)
#     print(f"Dropped collection: {collection_name}")

# Copy Users and Projects collections from source -> test databases


In [398]:
users_collection = db_source['users']
users = list(users_collection.find())
projects_collection = db_source['projects']
projects = list(projects_collection.find())

users_copy = db_copy['users']
projects_copy = db_copy['projects']

try:
    users_copy.insert_many(users, ordered=False) # Copy source db users to test db users
    projects_copy.insert_many(projects, ordered=False) # Copy source db projects to test db projects
except BulkWriteError as bwe:
    print("BulkWriteError details:")
    print(bwe.details)  # This contains info on which documents failed and why

# Get Users with at least one managedProjects

Retrieve a list of all users with at least one managedProject.


In [399]:
query = {
  "managedProjects": { 
      "$exists": True, 
      "$not": { "$size": 0 } 
  }
}

target_users = list(users_copy.find(query))

# Create an dictionary called `projects_users`

The dict has project IDs as keys and arrays of user IDs as values


In [400]:
projects_users = {}

# Function to filter only projects with valid mongoose IDs
def filter_valid_mongoose_ids(id_list):
    return [x for x in id_list if ObjectId.is_valid(x)]

for user in target_users:
    # Destructure id and managed projects from user
    _id, managed_projects = user['_id'], user['managedProjects']

    # Filter projects
    filtered_projects = filter_valid_mongoose_ids(managed_projects)

    for proj_id in filtered_projects:
        if proj_id in projects_users:
            projects_users[f"{proj_id}"].append(_id)
        else:
            projects_users[f"{proj_id}"] = [_id]

pp.pprint(projects_users)

{'68a3e64ee2653c001fe3ff3b': [ObjectId('6481155fab091f001e30925b'),
                              ObjectId('66024c13e6a0050028e07948'),
                              ObjectId('670dd397cace6a002abb20ce')],
 '68a3e75ea19d60385b3938f8': [ObjectId('670dd397cace6a002abb20ce')]}


# Update `managedByUsers` field in Projects 

Update all project's `managedByUsers` array using bulk write

In [404]:
operations = []

for proj_id, user_ids in projects_users.items():
    valid_user_ids = [uid for uid in user_ids if ObjectId.is_valid(uid)]    

    proj = projects_copy.find_one({"_id": ObjectId(proj_id)})

    if proj:
        print('Project before update:')
        pp.pprint(proj)
        
        # Compile individual updates in operations 
        operations.append(UpdateOne(
            {"_id": ObjectId(proj_id)}, # Filter
            {"$set": {"managedByUsers": valid_user_ids}}, # Update
        ))
    else:
        print(f"No project with {proj_id} found")

# Execute the bulk write to update operations
result = projects_copy.bulk_write(operations)

print(f"Result: ", result)

Project before update:
{'__v': 0,
 '_id': ObjectId('68a3e64ee2653c001fe3ff3b'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 49, 50, 843000),
 'description': 'Testing...',
 'githubIdentifier': 'lkjlkj',
 'githubUrl': 'lkjlk',
 'googleDriveUrl': 'https://drive.google.com/drive/folders/1hAq0wyZKOaZLujqOYiaFv5PYgooISger?usp=drive_link',
 'hflaWebsiteUrl': 'lkjlkj',
 'managedByUsers': [ObjectId('6481155fab091f001e30925b'),
                    ObjectId('66024c13e6a0050028e07948'),
                    ObjectId('670dd397cace6a002abb20ce')],
 'name': 'Jacks Test Project',
 'partners': [],
 'projectStatus': 'Active',
 'recruitingCategories': [],
 'slackUrl': 'lkjlkj'}
Project before update:
{'__v': 0,
 '_id': ObjectId('68a3e75ea19d60385b3938f8'),
 'createdDate': datetime.datetime(2025, 8, 19, 2, 54, 22, 871000),
 'description': 'afk',
 'githubIdentifier': 'afk',
 'githubUrl': 'afk',
 'googleDriveUrl': 'https://drive.google.com/test',
 'hflaWebsiteUrl': 'afk',
 'managedByUsers': [ObjectId('